# GenAI-Traces: Core Tracing Functionality

This notebook demonstrates the core tracing functionality of GenAI-Traces:
- Initializing the tracer
- Creating spans with context managers
- Using decorators
- Context propagation
- Exporting traces

In [ ]:
# Add parent directory to path for development
import sys
sys.path.insert(0, '..')

## 1. Initialize the Tracer

In [ ]:
from genai_traces import init_tracer, get_tracer
from genai_traces.exporters import ConsoleExporter, JSONFileExporter

# Initialize with console exporter for visibility
tracer = init_tracer(
    service_name="notebook-demo",
    environment="development",
    exporters=[ConsoleExporter(pretty=True, color=True)],
)

print(f"Tracer initialized: {tracer.config.service_name}")

## 2. Basic Span Creation with Context Manager

In [ ]:
from genai_traces.core.types import SpanType

# Create a simple span
with tracer.start_as_current_span("my_first_span", SpanType.LLM) as span:
    span.set_attribute("custom.attribute", "hello world")
    span.set_attribute("llm.model.name", "gpt-4o")
    
    # Simulate some work
    import time
    time.sleep(0.1)
    
    span.set_attribute("llm.completion", "This is a test response.")

print("Span completed!")

## 3. Nested Spans (Parent-Child Relationships)

In [ ]:
with tracer.start_as_current_span("parent_workflow", SpanType.WORKFLOW) as parent:
    parent.set_attribute("workflow.name", "document_processing")
    
    # Child span 1: Retrieval
    with tracer.start_as_current_span("retrieve_documents", SpanType.RETRIEVAL) as child1:
        child1.set_attribute("retrieval.count", 5)
        time.sleep(0.05)
    
    # Child span 2: LLM Call
    with tracer.start_as_current_span("generate_summary", SpanType.LLM) as child2:
        child2.set_attribute("llm.model.name", "gpt-4o")
        child2.set_attribute("llm.prompt", "Summarize the documents...")
        time.sleep(0.05)
        child2.set_attribute("llm.completion", "Summary: ...")

print("\nNested spans completed!")
print(f"Parent trace_id: {parent.trace_id}")

## 4. Using Decorators

In [ ]:
from genai_traces import trace, trace_llm, trace_tool

@trace_llm(model="gpt-4o", provider="openai")
def generate_text(prompt: str) -> str:
    """Simulated LLM call."""
    time.sleep(0.05)
    return f"Response to: {prompt}"

@trace_tool()
def search_web(query: str) -> list:
    """Simulated web search."""
    time.sleep(0.03)
    return [f"Result 1 for {query}", f"Result 2 for {query}"]

@trace(span_type="agent", name="research_agent")
def run_research(topic: str) -> str:
    """Research agent that uses tools and LLM."""
    results = search_web(topic)
    summary = generate_text(f"Summarize: {results}")
    return summary

# Run the agent
result = run_research("AI trends 2024")
print(f"\nAgent result: {result}")

## 5. Async Tracing

In [ ]:
import asyncio

async def async_llm_call(prompt: str) -> str:
    async with tracer.start_as_current_span_async("async_llm", SpanType.LLM) as span:
        span.set_attribute("llm.prompt", prompt)
        await asyncio.sleep(0.05)
        response = f"Async response to: {prompt}"
        span.set_attribute("llm.completion", response)
        return response

# Run async function
result = await async_llm_call("What is the meaning of life?")
print(f"\nAsync result: {result}")

## 6. Error Handling

In [ ]:
try:
    with tracer.start_as_current_span("error_span", SpanType.LLM) as span:
        span.set_attribute("llm.prompt", "This will fail")
        raise ValueError("Simulated API error")
except ValueError as e:
    print(f"Caught error: {e}")
    print("Error was recorded in the span!")

## 7. Conversation Context

In [ ]:
from genai_traces import set_conversation_context

# Set conversation context
set_conversation_context(
    conversation_id="conv_12345",
    turn=1,
    user_id="user_abc"
)

# Spans will now automatically include conversation context
with tracer.start_as_current_span("chat_turn_1", SpanType.CHAT) as span:
    span.set_attribute("llm.prompt", "Hello!")
    span.set_attribute("llm.completion", "Hi there! How can I help?")

print("Conversation context attached to span!")

## Summary

This notebook demonstrated:
- ✅ Tracer initialization
- ✅ Basic span creation
- ✅ Nested spans with parent-child relationships
- ✅ Decorators for automatic tracing
- ✅ Async tracing support
- ✅ Error handling and recording
- ✅ Conversation context propagation